# Proyecto Final · Ciencia de Datos
## Segmentación de tickers bursátiles mediante clustering no supervisado

**Autora:** Ashly Geraldine Sotelo Alonso
**Curso:** Especialidad en Ciencia de Datos · Cohorte 3 (Bancolombia · Esumer)
**Fecha de entrega:** Mayo de 2026

### Resumen ejecutivo

El mercado bursátil ofrece miles de activos con perfiles de riesgo y rentabilidad muy diferentes,
lo que hace inviable analizarlos uno a uno cuando se quiere construir una recomendación de inversión.
Este proyecto aborda ese problema sobre una muestra de **20 acciones del mercado estadounidense**
con seis meses de precios diarios y volúmenes de transacción, y aplica técnicas de **aprendizaje
no supervisado** (K-means y, como contraste, DBSCAN) para descubrir grupos de tickers que se
comportan de forma similar.

A partir de tres métricas financieras — **rentabilidad media diaria**, **volatilidad** y **volumen
promedio** — se construye un perfil cuantitativo por ticker. Sobre estos perfiles se ejecuta el
clustering, se reduce la dimensionalidad con PCA para visualizar los grupos en 2D y, finalmente,
se asignan etiquetas interpretativas (defensivo, especulativo, popular, mixto) que se traducen en
recomendaciones para distintos perfiles de inversor (estabilidad, crecimiento, liquidez).

El entregable consiste en este mismo notebook como **informe ejecutable**: cada paso está
acompañado de su justificación, sus visualizaciones y la lectura cualitativa de los resultados.

### Objetivos

**Objetivo general**

Identificar grupos de acciones bursátiles que comparten comportamientos financieros similares
mediante técnicas de agrupamiento no supervisado, y extraer conclusiones valiosas para la
toma de decisiones de inversión.

**Objetivos específicos**

1. Cargar y describir el dataset de precios y volúmenes diarios de 20 tickers a lo largo de
   aproximadamente seis meses.
2. Calcular para cada ticker tres métricas resumen — rentabilidad media diaria, volatilidad y
   volumen promedio — que servirán como espacio de características del clustering.
3. Diagnosticar la calidad del dataset (valores nulos, *outliers*) y aplicar la limpieza
   justificada que sea necesaria.
4. Estandarizar las métricas y aplicar el algoritmo **K-means** seleccionando el número óptimo
   de clústeres mediante el **método del codo** y el **índice de silueta**, complementado con
   **DBSCAN** como contraste basado en densidad.
5. Reducir las métricas escaladas a dos dimensiones con **PCA** para visualizar la separación
   de los grupos.
6. Interpretar cada clúster en términos financieros, asignándole una etiqueta cualitativa y
   un **perfil de inversor** asociado, y formular recomendaciones de inversión.

## 1. Recolección de datos

> 💡 **¿Qué dataset estamos analizando?**
> El archivo `precios_consolidados_tickers.csv` contiene la información diaria de **20 acciones
> del mercado estadounidense** durante un período aproximado de **seis meses** (250 días hábiles).
> El formato es *wide*: por cada ticker hay dos columnas, `<TICKER>_Close` (precio de cierre)
> y `<TICKER>_Volume` (volumen transado). Esta estructura nos permite calcular métricas de
> forma vectorizada y comparar fácilmente el comportamiento de los activos en el mismo eje
> temporal.

### 1.1 Importación de bibliotecas

Importamos en un único bloque toda la pila que utilizaremos a lo largo del notebook, agrupada
por uso: manipulación de datos (`pandas`, `numpy`), visualización (`matplotlib`, `seaborn`),
estadística descriptiva (`scipy.stats`) y aprendizaje no supervisado (`scikit-learn`).
Fijamos también la semilla aleatoria y el estilo gráfico por defecto para que todos los
resultados del notebook sean reproducibles.

In [1]:
# Manipulación y análisis de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística descriptiva (asimetría, curtosis, etc.)
from scipy import stats

# Aprendizaje no supervisado
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Configuración global de estilo y reproducibilidad
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13

RANDOM_STATE = 42  # se usará en KMeans, PCA y cualquier proceso estocástico

### 1.2 Carga del archivo

Leemos el CSV indicando que la columna `Fecha` debe interpretarse como tipo *datetime* y la
fijamos como índice del `DataFrame`. Trabajar con un índice temporal es la convención estándar
en análisis financiero: simplifica filtrar rangos de fechas, ordenar y aplicar operaciones
desplazadas como `pct_change()`.

In [2]:
df = pd.read_csv("precios_consolidados_tickers.csv", parse_dates=["Fecha"])
df.set_index("Fecha", inplace=True)
df.head()

,AMZN_Close,AMZN_Volume,AMD_Close,AMD_Volume,AAPL_Close,AAPL_Volume,ASML_Close,ASML_Volume,BRKb_Close,BRKb_Volume,...,NNND_Close,NNND_Volume,TSLA_Close,TSLA_Volume,TM_Close,TM_Volume,V_Close,V_Volume,WMT_Close,WMT_Volume
Fecha,,,,,,,,,,,,,,,,,,,,,
2024-05-07,188.76,34050000.0,154.43,37370000.0,182.40,77310000.0,908.22,655000.0,406.14,3090000.0,...,365.8,19410000.0,177.81,75050000.0,231.26,248940.0,276.46,6380000.0,60.62,14520000.0
2024-05-08,188.00,26140000.0,153.62,28730000.0,182.74,45060000.0,911.47,555750.0,406.37,2400000.0,...,361.4,20060000.0,174.72,79970000.0,231.78,371660.0,277.19,9030000.0,60.30,11020000.0
2024-05-09,189.50,43370000.0,152.39,33020000.0,184.57,48980000.0,913.54,755990.0,408.82,2360000.0,...,369.8,16030000.0,171.97,65950000.0,227.24,329110.0,278.54,8950000.0,60.44,14550000.0
2024-05-10,187.48,34140000.0,151.92,37650000.0,183.05,50760000.0,930.29,814080.0,412.05,3090000.0,...,371.0,15690000.0,168.47,72630000.0,218.78,563940.0,280.74,8990000.0,60.48,12360000.0
2024-05-13,186.57,24900000.0,150.56,27860000.0,186.28,72040000.0,917.24,745780.0,411.22,2710000.0,...,378.2,20560000.0,171.89,67020000.0,215.64,466170.0,279.39,10530000.0,60.41,19260000.0


### 1.3 Identificación de los tickers

En lugar de "quemar" la lista de los 20 tickers en el código, los **derivamos dinámicamente** a
partir de las columnas que terminan en `_Close`. Esta práctica es más robusta: si en el futuro
el archivo trajera más o menos activos, el resto del notebook seguiría funcionando sin tocar
nada. También sirve como verificación implícita de que el dataset tiene la estructura esperada.

In [3]:
# Columnas de cierre → nombres de los tickers
columnas_close = [col for col in df.columns if col.endswith("_Close")]
tickers = [col.split("_")[0] for col in columnas_close]

print(f"Número de tickers detectados: {len(tickers)}")
print(f"Tickers: {tickers}")

Número de tickers detectados: 20
Tickers: ['AMZN', 'AMD', 'AAPL', 'ASML', 'BRKb', 'KO', 'JPM', 'MA', 'META', 'MSFT', 'NESN', 'NVDA', 'PG', 'PM', 'TSM', 'NNND', 'TSLA', 'TM', 'V', 'WMT']


In [4]:
# Visión rápida del tamaño y rango temporal
print(f"Forma del DataFrame: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Rango de fechas:    {df.index.min().date()}  →  {df.index.max().date()}")
print(f"Total de días:      {len(df)} días hábiles")

Forma del DataFrame: 250 filas × 40 columnas
Rango de fechas:    2024-05-07  →  2025-05-06
Total de días:      250 días hábiles


**Observaciones iniciales sobre la estructura:**

- El dataset cubre 250 días hábiles desde mayo de 2024 hasta inicios de mayo de 2025,
  consistente con los ~6 meses anunciados en la guía (la cifra se acerca más a un año natural
  porque los días no hábiles no aparecen).
- Aparecen los 20 tickers esperados, con representación principalmente del mercado
  estadounidense pero también activos cotizados en otras plazas — por ejemplo `NESN`
  (Nestlé, Suiza), `ASML` (Países Bajos), `TM` (Toyota, Japón) y `NNND` (Nintendo). Esto será
  relevante en la siguiente sección porque sus calendarios bursátiles no coinciden 100% con
  el estadounidense y veremos algunos huecos.
- Vale la pena notar la convención de naming: **`BRKb`** (Berkshire Hathaway clase B) usa la
  letra `b` minúscula, no `BRK.B` ni `BRK-B` como suele verse en otras fuentes. Mantendremos
  esa misma grafía a lo largo del notebook.
- Por ahora no entramos en el diagnóstico de valores nulos ni en la limpieza: ese trabajo
  corresponde a la **§3** y conviene hacerlo después de calcular las métricas para que la
  decisión esté informada por lo que vamos a observar.

## 2. Cálculo de métricas por ticker

En esta sección transformamos la serie temporal de 250 días × 40 columnas en una **tabla
compacta de 20 filas (una por ticker) × 3 columnas** que resume el comportamiento financiero
de cada activo. Esa tabla — y no la serie original — será el insumo del clustering en §4.

Adicionalmente, calculamos un segundo conjunto de **métricas descriptivas** (coeficiente de
variación, asimetría y curtosis) que **no entran al clustering** pero enriquecen el análisis
cualitativo, en línea con lo trabajado en el Encuentro 8 (Estadística 1) sobre series de
acciones.

### 2.1 Conceptos clave

> 💡 **Las tres métricas que vamos a usar para segmentar**
>
> - **Rentabilidad media diaria** ($\bar{R}$): promedio de los retornos porcentuales día a día,
>   $R_t = (P_t - P_{t-1}) / P_{t-1}$. Es una proxy del *retorno esperado* del activo en escala
>   diaria. Valores positivos indican que en promedio la acción sube; negativos, lo contrario.
>
> - **Volatilidad** ($\sigma_R$): desviación estándar de los mismos retornos diarios. Mide qué
>   tanto se aleja el retorno típico de su media — es la medida estándar de **riesgo** en
>   finanzas cuantitativas.
>
> - **Volumen promedio** ($\bar{V}$): media de la cantidad de acciones transadas por día.
>   Es una proxy de **liquidez**: tickers con volumen alto se pueden comprar y vender sin
>   mover demasiado el precio, los de volumen bajo no.
>
> Todas las métricas se calculan en **escala diaria**, sin anualizar, que es la convención
> usada en el curso.

### 2.2 Construcción del DataFrame de métricas

Aislamos los precios de cierre y los volúmenes en dos `DataFrames` separados, renombrando las
columnas para que queden únicamente con el ticker (sin el sufijo `_Close` o `_Volume`).
Luego derivamos los retornos diarios con `pct_change()` y agregamos las tres métricas en un
único `DataFrame` indexado por ticker, que llamaremos `metricas`.

In [5]:
# Aislamos los precios de cierre, renombrando las columnas a solo el ticker.
precios = df[columnas_close].copy()
precios.columns = tickers

# Hacemos lo mismo con los volúmenes.
columnas_volume = [f"{t}_Volume" for t in tickers]
volumenes = df[columnas_volume].copy()
volumenes.columns = tickers

# Retornos diarios: variación porcentual día a día.
# pct_change() introduce un NaN en la primera fila (no hay día previo);
# las funciones .mean() y .std() lo ignoran automáticamente.
retornos = precios.pct_change()

retornos.head()

,AMZN,AMD,AAPL,ASML,BRKb,KO,JPM,MA,META,MSFT,NESN,NVDA,PG,PM,TSM,NNND,TSLA,TM,V,WMT
Fecha,,,,,,,,,,,,,,,,,,,,
2024-05-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-05-08,-0.004026,-0.005245,0.001864,0.003578,0.000566,NaN,0.020339,0.003774,0.009311,0.002932,0.022135,-0.001546,-0.004163,0.007382,0.017646,-0.012028,-0.017378,0.002249,0.002641,-0.005279
2024-05-09,0.007979,-0.008007,0.010014,0.002271,0.006029,NaN,0.009456,0.001583,0.005967,0.004336,NaN,-0.018361,0.005876,0.012316,-0.005641,0.023243,-0.015739,-0.019588,0.004870,0.002322
2024-05-10,-0.010660,-0.003084,-0.008235,0.018335,0.007901,NaN,0.006430,0.003271,0.001641,0.005869,NaN,0.012732,0.004878,0.002011,0.045311,0.003245,-0.020352,-0.037229,0.007898,0.000662
2024-05-13,-0.004854,-0.008952,0.017645,-0.014028,-0.002014,NaN,-0.000201,0.001707,-0.017199,-0.002459,-0.001891,0.005785,-0.005874,-0.003713,-0.019094,0.019407,0.020300,-0.014352,-0.004809,-0.001157


In [6]:
metricas = pd.DataFrame({
    "rentabilidad": retornos.mean(),
    "volatilidad": retornos.std(),
    "volumen_prom": volumenes.mean(),
})
metricas.index.name = "ticker"
metricas

,rentabilidad,volatilidad,volumen_prom
ticker,,,
AMZN,0.000142,0.021139,4.168388e+07
AMD,-0.001259,0.033200,4.227648e+07
AAPL,0.000546,0.020436,5.618684e+07
ASML,-0.000775,0.029893,1.641861e+06
BRKb,0.001010,0.012427,4.225040e+06
KO,0.002440,0.010565,1.804571e+07
JPM,0.001219,0.018227,9.646440e+06
MA,0.000931,0.013218,2.609420e+06
META,0.001168,0.022834,1.445908e+07


**Lectura preliminar de la tabla** (un análisis más profundo va en §6):

- Las **escalas son muy distintas** entre las tres métricas: rentabilidad y volatilidad están
  en el orden de $10^{-3}$ (milésimas), mientras que el volumen está en el orden de $10^6$ a
  $10^8$. Esto es exactamente la razón por la que en §4 vamos a **estandarizar** las variables
  antes de aplicar K-means: sin escalar, el volumen dominaría la distancia euclidiana y los
  clusters se formarían básicamente sobre esa única variable.
- A simple vista ya destacan tickers con perfiles bien diferenciados: `NVDA` con rentabilidad
  alta, `KO` con cifras que se ven raras (volumen muy bajo, casi seguro porque tiene muchos
  vacíos en el archivo — lo confirmaremos en §3), y nombres como `AMZN`, `TSLA` o `NVDA` con
  volúmenes muy superiores al resto.
- La fila `KO` es el principal punto de atención: dejamos su diagnóstico y la decisión sobre
  conservarla o eliminarla para la siguiente sección.

### 2.3 Métricas descriptivas adicionales

> 💡 **Más allá de la media y la desviación estándar** *(referencia: Encuentro 8 — Estadística 1)*
>
> La rentabilidad y la volatilidad describen el **centro** y la **dispersión** de la
> distribución de retornos, pero pierden información sobre su **forma**. Tres métricas
> adicionales nos ayudan a complementar la lectura:
>
> - **Coeficiente de variación** ($CV = \sigma / |\bar{R}|$): cuántas unidades de riesgo se
>   asumen por cada unidad de retorno medio. Es una forma de "volatilidad normalizada" que
>   permite comparar activos con rentabilidades muy distintas. Tomamos el valor absoluto de
>   $\bar{R}$ en el denominador para que el CV siga siendo interpretable cuando la
>   rentabilidad media es negativa.
>
> - **Asimetría (*skewness*)**: mide la desviación de la distribución respecto a la simetría.
>   Positiva → cola larga hacia la derecha (días de retornos extremos al alza más frecuentes);
>   negativa → cola larga a la izquierda (los días de pérdida extrema dominan).
>
> - **Curtosis (*kurtosis*)**: mide cuán "puntiaguda" es la distribución y qué tanto pesan
>   sus colas. En finanzas, una curtosis alta indica retornos extremos (positivos o negativos)
>   más frecuentes que en una normal — es una señal clásica de riesgo de cola.
>
> Estas tres métricas las usaremos solo para **enriquecer la interpretación** en §6;
> **no entran como variables del clustering** para no introducir redundancia con la
> volatilidad y para mantenernos cercanos al alcance de la guía.

In [7]:
# Coeficiente de variación: σ / |R̄|.
# Usamos valor absoluto en el denominador para que el indicador sea
# interpretable también cuando la rentabilidad media es negativa.
cv = retornos.std() / retornos.mean().abs()

metricas_extra = pd.DataFrame({
    "cv": cv,
    "skewness": retornos.skew(),
    "kurtosis": retornos.kurt(),
})
metricas_extra.index.name = "ticker"
metricas_extra

,cv,skewness,kurtosis
ticker,,,
AMZN,148.942118,0.118332,5.609553
AMD,26.371513,1.035680,10.795591
AAPL,37.445959,0.979050,14.347348
ASML,38.569270,-0.244457,6.900829
BRKb,12.299618,-0.373570,7.523022
KO,4.328984,0.155091,-0.605839
JPM,14.957002,0.384576,9.328288
MA,14.191006,-0.341125,8.426477
META,19.550597,0.597377,7.234301


**Lectura preliminar de las métricas extra:**

- Los **CV** muy altos (en valor absoluto) corresponden a tickers cuya rentabilidad media
  cercana a cero hace que el ratio se infle: no significa necesariamente "más riesgo", sino
  que "el retorno medio está muy cerca de cero relativo a su dispersión". Hay que leerlos con
  ese matiz.
- La mayoría de tickers presenta **curtosis positiva** (colas más pesadas que la normal), lo
  que es típico de los retornos diarios de acciones — confirma que las distribuciones no son
  gaussianas y que los modelos basados en supuesto de normalidad subestiman el riesgo
  extremo.
- La **asimetría** se reparte entre positiva y negativa según el ticker; la mayoría queda
  cerca de cero, lo que indica que en el período no hubo un sesgo claro hacia ganancias o
  pérdidas extremas predominantes.

Volveremos a estas métricas en la **§6**, cuando interpretemos cualitativamente los clústeres
formados.